<div dir="rtl" style="text-align: center; font-family: Arial, sans-serif; line-height: 1.6;">
    <h1 style="color: #2c3e50;">آشنایی با درخت تصمیم و جنگل تصادفی</h1>
</div>

<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h2>مقدمه</h2>
    <h3>درخت تصمیم (Decision Tree) چیست؟</h3>
    <p>
        درخت تصمیم یک مدل یادگیری ماشین قدرتمند و پرکاربرد است که هم برای مسائل طبقه‌بندی (Classification) و هم برای رگرسیون (Regression) استفاده می‌شود. این مدل با ساختاری شبیه به یک درخت، تصمیم‌گیری‌ها را شبیه‌سازی می‌کند.
    </p>
    <p>
        برای مثال:
        <ul>
            <li>در پزشکی، می‌توان از درخت تصمیم برای تشخیص بیماری بر اساس علائم بیمار استفاده کرد.</li>
            <li>در بانکداری، برای ارزیابی ریسک اعتبار مشتریان و تصمیم‌گیری در مورد اعطای وام به کار می‌رود.</li>
            <li>در بازاریابی، برای پیش‌بینی اینکه کدام مشتریان به یک کمپین تبلیغاتی پاسخ مثبت خواهند داد، استفاده می‌شود.</li>
        </ul>
    </p>
    <p>
        درخت تصمیم به ما کمک می‌کند تا الگوهای پیچیده در داده‌ها را به صورت مجموعه‌ای از قوانین ساده و قابل فهم نمایش دهیم. هر گره در درخت یک ویژگی (Feature) را آزمایش می‌کند و هر شاخه نشان‌دهنده نتیجه آن آزمایش است. برگ‌های درخت نیز برچسب کلاس یا مقدار پیش‌بینی‌شده را نشان می‌دهند.
    </p>
    <h3>جنگل تصادفی (Random Forest) چیست؟</h3>
    <p>
        جنگل تصادفی یک الگوریتم یادگیری گروهی (Ensemble Learning) است که با ساخت تعداد زیادی درخت تصمیم و ترکیب نتایج آن‌ها، دقت پیش‌بینی را به طور قابل توجهی بهبود می‌بخشد. این روش با کاهش مشکل بیش‌برازش (Overfitting) که در درخت‌های تصمیم تک رایج است، به یکی از محبوب‌ترین الگوریتم‌ها در یادگیری ماشین تبدیل شده است.
    </p>
    <p>
        در این دفترچه یادداشت، شما ابتدا یک مدل درخت تصمیم را از پایه پیاده‌سازی خواهید کرد و سپس با استفاده از آن، یک مدل جنگل تصادفی خواهید ساخت. در طول این مسیر، با مفاهیم کلیدی مانند ناخالصی جینی (Gini Impurity) و بهره اطلاعاتی (Information Gain) آشنا خواهید شد.
    </p>
</div>

<div dir="rtl" style="text-align: right;">

# پیاده‌سازی درخت تصمیم

در این بخش، گام به گام یک طبقه‌بند درخت تصمیم را پیاده‌سازی می‌کنیم. این درخت با دیتافریم‌های `pandas` کار می‌کند و قابلیت مدیریت ویژگی‌های عددی و دسته‌ای را خواهد داشت.

</div>

In [111]:
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
from typing import Any, Tuple, Union
from collections import Counter

<div dir="rtl" style="text-align: right;">

## بخش ۱: ساختار گره‌ها (Nodes)
هر درخت تصمیم از گره‌ها تشکیل شده است. ما سه نوع گره خواهیم داشت:
1.  **`Node`**: یک کلاس پایه که سایر گره‌ها از آن ارث‌بری می‌کنند.
2.  **`LeafNode`**: گره برگ که پیش‌بینی نهایی را انجام می‌دهد.
3.  **`NumericSplitNode`**: گره تصمیم که بر اساس یک ویژگی عددی و یک آستانه، داده‌ها را تقسیم می‌کند.
4.  **`CategoricalSplitNode`**: گره تصمیم که بر اساس یک ویژگی دسته‌ای، داده‌ها را تقسیم می‌کند.

</div>

In [112]:
class Node:
    """
    Base class for all nodes in the decision tree.
    This class is not meant to be instantiated directly.
    """
    def predict(self, x: pd.Series) -> Any:
        """
        Predicts the class for a single data point.
        This method should be overridden by subclasses.

        Args:
            x (pd.Series): A single data point (features), with index as feature names.

        Returns:
            Any: The predicted class label.
        """
        # TODO: This method must be implemented by child classes to define prediction logic.
        raise NotImplementedError("Subclasses should implement this method.")

In [113]:
class LeafNode(Node):
    """
    Represents a leaf node in the decision tree.
    This node makes a prediction based on the majority class of the training samples
    that reached this leaf.
    """
    def __init__(self, y: np.ndarray):
        """
        Initializes the LeafNode.

        Args:
            y (np.ndarray): The target values of the training samples at this leaf.
        """
        # TODO: Find the most frequent class in `y` and store it as this leaf's prediction.
        # You can use np.unique with return_counts=True, and np.argmax
        self.classes, self.count = np.unique(y, return_counts=True)
        self.prediction = self.classes[np.argmax(self.count)]


    def predict(self, x: pd.Series) -> Any:
        """
        Returns the stored prediction for any data point that reaches this leaf.

        Args:
            x (pd.Series): A single data point (features). Not used in a leaf node
                           but required for a consistent interface.

        Returns:
            Any: The predicted class label for this leaf.
        """
        return self.prediction

In [114]:
class NumericSplitNode(Node):
    """
    Represents a node that splits data based on a numeric feature and a threshold.
    """
    def __init__(self, feature_name: str, threshold: float, left_branch: Node, right_branch: Node):
        """
        Initializes the NumericSplitNode.

        Args:
            feature_name (str): The name of the numeric feature to split on.
            threshold (float): The value to compare the feature against.
            left_branch (Node): The child node for values <= threshold.
            right_branch (Node): The child node for values > threshold.
        """
        # TODO: Initialize class parameters
        self.feature = feature_name
        self.threshold = threshold
        self.left = left_branch
        self.right = right_branch
        
    def predict(self, x: pd.Series) -> Any:
        """
        Directs the prediction to the appropriate child node based on the feature value
        and threshold.

        Args:
            x (pd.Series): A single data point (features).

        Returns:
            Any: The prediction from the corresponding child node.
        """
        # TODO: Based on whether x's feature value is <= the threshold, recursively predict using the left or right branch.
        if x[self.feature] <= self.threshold:
            return self.left.predict(x)
        
        else:
            return self.right.predict(x)

In [115]:
class CategoricalSplitNode(Node):
    """
    Represents a node that splits data based on a categorical feature using a
    one-vs-all strategy.
    """
    def __init__(self, feature_name: str, category: Any, match_branch: Node, rest_branch: Node):
        """
        Initializes the CategoricalSplitNode.

        Args:
            feature_name (str): The name of the categorical feature.
            category (Any): The specific category to check for (the "one").
            match_branch (Node): The child node if the feature matches the category.
            rest_branch (Node): The child node if the feature does not match (the "all").
        """
        # TODO: Initialize class parameters
        self.feature = feature_name
        self.category = category
        self.match = match_branch
        self.rest = rest_branch

    def predict(self, x: pd.Series) -> Any:
        """
        Directs the prediction to the appropriate child node based on whether the
        feature value matches the specified category.

        Args:
            x (pd.Series): A single data point (features).

        Returns:
            Any: The prediction from the corresponding child node.
        """
        # TODO: Based on whether x's feature value matches the category, recursively predict using the match or rest branch.
        if x[self.feature] == self.category:
            return self.match.predict(x)
        
        else:
            return self.rest.predict(x)


<div dir="rtl" style="text-align: right;">

## بخش ۲: پیاده‌سازی درخت تصمیم
حالا که ساختار گره‌ها را داریم، می‌توانیم کلاس `DecisionTree` را بسازیم. این کلاس شامل منطق ساخت درخت (آموزش) و پیش‌بینی خواهد بود.

### بحث کنید
- **شرایط توقف**: چرا باید برای رشد درخت شرایط توقف مانند `max_depth` (حداکثر عمق) و `min_samples_split` (حداقل نمونه برای تقسیم) در نظر بگیریم؟ اگر این کار را نکنیم چه اتفاقی می‌افتد؟

</div>

In [131]:
class DecisionTree:
    """
    A decision tree classifier that works with Pandas DataFrames.

    Attributes:
        min_samples_split (int): The minimum number of samples required to split a node.
        max_depth (int): The maximum depth of the tree.
        root (Node): The root node of the decision tree after fitting.
    """
    def __init__(self, min_samples_split: int = 10, max_depth: int = 10):
        """
        Initializes the DecisionTree.

        Args:
            min_samples_split (int): The minimum number of samples a node must have
                                     to be considered for splitting.
            max_depth (int): The maximum depth allowed for the tree.
        """
        # TODO: Initialize class parameters
        self.min_sample_split = min_samples_split
        self.max_depth = max_depth

    def _gini(self, y: np.ndarray) -> float:
        """
        Calculates the Gini impurity for a set of labels.

        Args:
            y (np.ndarray): An array of labels.

        Returns:
            float: The Gini impurity score.
        """
        # TODO: Calculate Gini impurity using the formula: 1 - sum of squared probabilities of each class.
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / counts.sum()
        return 1.0 - np.sum(probabilities ** 2)

    def _find_best_numeric_split(self, feature_values: np.ndarray, y: np.ndarray, parent_gini: float) -> Tuple[float, float]:
        """
        Finds the best split for a single numeric feature's values.

        Args:
            feature_values (np.ndarray): The values of the feature to split on.
            y (np.ndarray): The target labels.
            parent_gini (float): The Gini impurity of the parent node.

        Returns:
            best_gain (float): The best information gain.
            best_threshold (Any): The best threshold to split on.
        
        Returns (-1.0, None) if no split is found.
        """
        # TODO: Iterate through sorted unique feature values, calculate info gain for each split, and return the best one.
        sorted_ind = np.argsort(feature_values)
        feature_values = feature_values.to_numpy()[sorted_ind]
        y = np.array(y)
        y = y[sorted_ind]

        best_gain = -1.0
        best_threshold = None

        for i in range(1, len(feature_values)):
            if feature_values[i] == feature_values[i - 1]:
                continue
            
            threshold = (feature_values[i] + feature_values[i - 1]) / 2
            left = feature_values <= threshold
            right = ~left

            if sum(left) == 0 or sum(right) == 0:
                continue

            gini_left = self._gini(y[left])
            gini_right = self._gini(y[right])
            weighted_gini = (gini_left * left.sum() + gini_right * right.sum()) / len(y)

            gain = parent_gini - weighted_gini
            if gain > best_gain:
                best_gain = gain
                best_threshold = threshold

        return best_gain, best_threshold

    def _find_best_categorical_split(self, feature_values: pd.Series, y: np.ndarray, parent_gini: float) -> Tuple[float, Any]:
        """
        Finds the best one-vs-rest split for a single categorical feature.

        Args:
            feature_values (pd.Series): The values of the feature to split on.
            y (np.ndarray): The target labels.
            parent_gini (float): The Gini impurity of the parent node.

        Returns:
            best_gain (float): The best information gain.
            best_category (Any): The best category to split on.
        
        Returns (-1.0, None) if no split is found.
        """
        # TODO: Iterate through unique categories, calculate info gain for each one-vs-rest split, and return the best.
        best_gain = -1.0
        best_category = None

        for category in np.unique(feature_values):
            left = (feature_values == category)
            right = ~left

            if left.sum() == 0 or right.sum() == 0:
                continue

            gini_left = self._gini(y[left])
            gini_right = self._gini(y[right])

            n = len(y)
            n_left = left.sum()
            n_right = right.sum()

            weighted_gini = (n_left / n) * gini_left + (n_right / n) * gini_right
            gain = parent_gini - weighted_gini

            if gain > best_gain:
                best_gain = gain
                best_category = category

        return best_gain, best_category

    def _find_best_split(self, X: pd.DataFrame, y: np.ndarray) -> Tuple[str, Any]:
        """
        Finds the best feature and value/category to split on by maximizing information gain.

        Args:
            X (pd.DataFrame): The input data for the current node.
            y (np.ndarray): The target labels for the current node.

        Returns:
            best_feature_name (str): A tuple containing the name of the best feature to split on.
            best_split_value (Any): the threshold/category for the split.
            
        Returns (None, None) if no profitable split is found.
        """
        # TODO: Iterate through all features, find the best split for each, 
        # and return the feature and split value with the highest info gain.
        parent_gini = self._gini(y)
        best_gain = -1.0
        best_feature = None
        best_split = None

        for col in X.columns:
            feature_values = X[col]

            if is_numeric_dtype(feature_values):
                gain, threshold = self._find_best_numeric_split(feature_values, y, parent_gini)

            else:
                gain, threshold = self._find_best_categorical_split(feature_values, y, parent_gini)

            if gain > best_gain:
                best_gain = gain
                best_feature = col
                best_split = threshold

        return best_feature, best_split

    def _grow_tree(self, X: pd.DataFrame, y: np.ndarray, depth: int = 0) -> Node:
        """
        Recursively grows the decision tree.

        Args:
            X (pd.DataFrame): The input data for the current node.
            y (np.ndarray): The target labels for the current node.
            depth (int): The current depth of the node in the tree.

        Returns:
            Node: The root node of the constructed subtree.
        """
        # TODO: Check stopping conditions; if met return a LeafNode, otherwise find the best split and recursively call this function on the resulting subsets.
        if len(y) < self.min_sample_split or depth >= self.max_depth or len(np.unique(y)) == 1:
            return LeafNode(y)
        
        best_feature, best_value = self._find_best_split(X, y)

        if best_feature is None:
            return LeafNode(y)
        
        feature_value = X[best_feature]
        if is_numeric_dtype(feature_value):
            left = feature_value <= best_value
            right = feature_value > best_value
            NodeClass = NumericSplitNode
        else:
            left = feature_value == best_value
            right = feature_value != best_value
            NodeClass = CategoricalSplitNode

        left_child = self._grow_tree(X[left], y[left], depth + 1)
        right_child = self._grow_tree(X[right], y[right], depth + 1)

        return NodeClass(best_feature, best_value, left_child, right_child)

    def fit(self, X: pd.DataFrame, y: np.ndarray) -> None:
        """
        Builds the decision tree from the training set (X, y).

        Args:
            X (pd.DataFrame): Training data.
            y (np.ndarray): Target values.
        """
        # TODO: Initiate the recursive tree growing process by calling `_grow_tree`.
        self.root = self._grow_tree(X, y, depth=0) 

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """
        Predicts class labels for samples in X.

        Args:
            X (pd.DataFrame): The input samples.

        Returns:
            np.ndarray: An array of predicted class labels.
        """
        # TODO: For each row in X, traverse the tree from the root to get a prediction, and return all predictions.
        return np.array([self.root.predict(row) for _, row in X.iterrows()])


<div dir="rtl" style="text-align: right;">

## بخش ۳: ارزیابی مدل‌ها
در این بخش، مدل‌هایی که ساختیم را بر روی دو مجموعه داده واقعی ارزیابی می‌کنیم: مجموعه داده تایتانیک و مجموعه داده سرطان پستان.
ابتدا داده‌ها را بارگذاری و آماده می‌کنیم.

</div>

<div dir="rtl" style="text-align: right;">

#### مجموعه داده تایتانیک
در این مجموعه داده، هدف پیشبینی زنده ماندن افراد درون کشتی تایتانیک است
</div>

In [132]:
# TODO: Read data from "titanic.csv"
df = pd.read_csv('./titanic.csv')
# TODO: Print the top rows
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [133]:
# Preprocess
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

# TODO: Select features and target from the dataframe
df = df[features + [target]]

# TODO: Fill missing values in 'Age' column with its medians.
# and 'Embarked' column with the most frequent value (You can use pd.Series.mode method).
df['Age'].fillna(int(df['Age'].median()), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# TODO Convert some of types to categorial
df['Pclass'] = df['Pclass'].astype('category')
df['Sex'] = df['Sex'].astype('category')
df['Embarked'] = df['Embarked'].astype('category')
df.dtypes

C:\Users\Radin\AppData\Local\Temp\ipykernel_14532\517073370.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(int(df['Age'].median()), inplace=True)
C:\Users\Radin\AppData\Local\Temp\ipykernel_14532\517073370.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.



Pclass      category
Sex         category
Age          float64
SibSp          int64
Parch          int64
Fare         float64
Embarked    category
Survived       int64
dtype: object

<div dir="rtl" style="text-align: right;">

#### مجموعه داده سرطان پستان
در این مجموعه داده، هدف پیشبینی وخیم بودن تومور است
</div>

In [134]:
# TODO: Read data from "breast_cancer.csv"
df_B = pd.read_csv('./breast_cancer.csv')
# TODO: Print dataset columns
df_B.columns

Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension',
       'is benign'],
      dtype='object')

<div dir="rtl" style="text-align: right;">

### تابع train_and_evaluate
از این تابع برای ارزیابی دقت مدل ها استفاده می کنیم
</div>

In [135]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def train_and_evaluate(dataframe: pd.DataFrame, target_column: str, model_class, **model_args):
    """
    Trains and evaluates a machine learning model.

    Args:
        dataframe (pd.DataFrame): The input dataframe containing features and target.
        target_column (str): The name of the target variable column.
        model_class: The classifier class to be instantiated (e.g., DecisionTree).
        **model_args: Arbitrary keyword arguments for the model's constructor.
    """
    # TODO: Select features (X) and target (y)
    X = dataframe.drop(columns=[target_column])
    y = dataframe[target_column].copy().values.ravel()
    
    # TODO: Separate training data and test data using "train_test_split"
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    
    # TODO: Instantiate the DecisionTree class and train the model using the training data.
    model = model_class(**model_args)
    
    # TODO: Make predictions on the test data.
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    
    # TODO Calculate the accuracy on test set using "accuracy_score"
    accuracy = accuracy_score(y_test, pred)
    
    # Print the accuracy and arguments
    print(f"--- Results for {model_class.__name__} ---")
    print("Model Arguments:")
    for key, value in model_args.items():
        print(f"  {key}: {value}")
    print(f"Accuracy: {accuracy:.4f}\n")
    
# TODO: call the function on titanic dataset
train_and_evaluate(df, target, DecisionTree)

--- Results for DecisionTree ---
Model Arguments:
Accuracy: 0.7982



<div dir="rtl" style="text-align: right;">

### ارزیابی درخت تصمیم
حالا تابع `train_and_evaluate` را با مدل `DecisionTree` خود بر روی هر دو مجموعه داده فراخوانی کنید.

</div>

In [136]:
# TODO: call train_and_evaluate function on titanic dataset with DecisionTree model
train_and_evaluate(df, target, DecisionTree, max_depth=6, min_samples_split=10)

--- Results for DecisionTree ---
Model Arguments:
  max_depth: 6
  min_samples_split: 10
Accuracy: 0.7758



In [137]:
# TODO: call train_and_evaluate function on titanic dataset with DecisionTree model
train_and_evaluate(df_B, 'is benign', DecisionTree, max_depth=7, min_samples_split=10)

--- Results for DecisionTree ---
Model Arguments:
  max_depth: 7
  min_samples_split: 10
Accuracy: 0.9441



<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h2>بخش 4: هرس درخت با استفاده از پیچیدگی هزینه (Cost-Complexity Pruning)</h2>
    <p>
        هرس کردن یک تکنیک حیاتی برای جلوگیری از بیش‌برازش (Overfitting) در درختان تصمیم است. یکی از روش‌های قدرتمند برای این کار، <b>هرس پیچیدگی هزینه</b> است که از یک پارامتر به نام <code>alpha</code> (α) برای ایجاد تعادل بین خطای طبقه‌بندی درخت و پیچیدگی آن (تعداد برگ‌ها) استفاده می‌کند.
    </p>
    <p>
        در این روش، ما ضعیف‌ترین شاخه (Weakest Link) را در درخت پیدا کرده و آن را هرس می‌کنیم. این کار تا زمانی ادامه می‌یابد که پیچیدگی درخت از یک مقدار مشخص شده توسط <code>alpha</code> کمتر شود. مقدار <code>alpha</code> بزرگتر منجر به هرس شدیدتر و درختی ساده‌تر می‌شود.
    </p>
    <p>
        <b>نکته:</b> برای پیاده‌سازی این توابع، فرض می‌شود که کلاس‌های <code>Node</code>، <code>LeafNode</code>، <code>NumericSplitNode</code> و <code>CategoricalSplitNode</code> از قبل تعریف شده‌اند و کلاس <code>LeafNode</code> مقادیر <code>y</code> نمونه‌های آموزشی که به آن برگ می‌رسند را در خود ذخیره می‌کند.
    </p>
</div>

<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h3 style="color: #4CAF50;">تابع کمکی: محاسبه آمار برای هرس</h3>
    <p>
        اولین گام، پیمایش درخت به صورت پس‌ترتیبی (Post-order) برای محاسبه و ذخیره سه مقدار کلیدی در هر گره است:
        <ul>
            <li><b>R(t)</b>: خطای گره اگر به یک برگ تبدیل شود.</li>
            <li><b>R(T_t)</b>: مجموع خطاهای زیردرخت منشعب از گره.</li>
            <li><b>|L(T_t)|</b>: تعداد برگ‌های زیردرخت.</li>
        </ul>
        این آمار برای محاسبه «پارامتر پیچیدگی» هر گره ضروری است.
    </p>
</div>

In [138]:
def _calculate_stats_for_pruning(node: Node) -> tuple:
    """
    Performs a post-order traversal to calculate stats needed for pruning.
    For each node `t`, it calculates:
    - R(t): The misclassification error of treating `t` as a leaf.
    - R(T_t): The total misclassification error of the subtree rooted at `t`.
    - |L(T_t)|: The number of leaves in the subtree rooted at `t`.
    
    These stats are stored as new attributes on each node object.
    
    Args:
        node: The current node in the traversal.

    Returns:
        A tuple of (y_values, subtree_error, leaf_count) for the given node.
    """
    # TODO: Traverse the tree in post-order to compute and store the error and leaf count for each node and its corresponding subtree.
    return None, None, None

<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h3 style="color: #4CAF50;">تابع کمکی: هرس بازگشتی</h3>
    <p>
        این تابع بازگشتی، پارامتر پیچیدگی <code>g(t)</code> را برای هر گره محاسبه می‌کند. اگر این مقدار کمتر یا مساوی <code>alpha</code> باشد، آن گره به یک برگ تبدیل می‌شود (هرس می‌شود). در غیر این صورت، تابع به صورت بازگشتی روی فرزندان آن گره فراخوانی می‌شود.
    </p>
</div>

In [139]:
def _prune_recursive_alpha(node: Node, alpha: float) -> Node:
    """
    Recursively prunes the tree. If a node's complexity g(t) is <= alpha,
    it is converted to a leaf.
    
    Args:
        node: The current node to consider pruning.
        alpha: The complexity parameter.

    Returns:
        The pruned node or subtree.
    """
    # TODO: Recursively check each node's complexity g(t). If g(t) <= alpha, prune the node by converting it to a leaf. Otherwise, continue the recursion on its children.
    return None

<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h3 style="color: #4CAF50;">تابع اصلی: هرس درخت</h3>
    <p>
        این تابع اصلی، دو تابع کمکی بالا را فراخوانی می‌کند. ابتدا آمار لازم را محاسبه کرده و سپس فرآیند هرس بازگشتی را با شروع از ریشه درخت آغاز می‌کند.
    </p>
</div>

In [140]:
def prune_tree_cost_complexity(root: Node, alpha: float) -> Node:
    """
    Prunes a decision tree using the Cost Complexity Pruning method.

    This function first traverses the tree to calculate the necessary error
    and complexity statistics for each node. It then recursively prunes nodes
    whose complexity parameter `g(t)` is less than or equal to the given `alpha`.

    Args:
        root (Node): The root node of the decision tree to be pruned.
                     The tree must have been trained, and LeafNodes must
                     contain the `y_values` of the training samples that fell
                     into them.
        alpha (float): The complexity parameter. A higher alpha results in
                       more pruning and a smaller tree. Alpha must be >= 0.

    Returns:
        Node: The root node of the new, pruned tree.
    """
    # TODO: First, populate the tree with pruning statistics by calling `_calculate_stats_for_pruning`. Then, initiate the recursive pruning process by calling `_prune_recursive_alpha`.
    return None

<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h2>بخش 4.5: ارزیابی درخت هرس شده</h2>
    <p>
        اکنون که توابع هرس را در اختیار داریم، تأثیر آن را بر عملکرد درخت تصمیم ارزیابی می‌کنیم. برای این کار، به یک مجموعه داده اعتبارسنجی (Validation Set) نیاز داریم تا پارامتر <code>alpha</code> را بر اساس آن تنظیم کرده و از بیش‌برازش جلوگیری کنیم. بنابراین، داده‌ها را به سه بخش تقسیم می‌کنیم: آموزش، اعتبارسنجی و آزمون.
    </p>
</div>

<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h3 style="color: #4CAF50;">تابع ارزیابی برای درخت هرس شده</h3>
    <p>
         این تابع جدید، داده‌ها را به سه بخش تقسیم کرده، مدل را آموزش می‌دهد، آن را هرس می‌کند و در نهایت دقت مدل را قبل و بعد از هرس بر روی مجموعه آزمون مقایسه می‌کند.
    </p>
</div>

In [141]:
def train_evaluate_and_prune(dataframe: pd.DataFrame, target_column: str, alpha: float, **model_args):
    """
    Trains, prunes, and evaluates a Decision Tree model.

    This function splits the data into training, validation, and test sets,
    trains a decision tree, evaluates its performance, prunes it using the
    provided alpha, and then evaluates the pruned tree's performance.

    Args:
        dataframe (pd.DataFrame): The input dataframe containing features and the target.
        target_column (str): The name of the target variable column.
        alpha (float): The complexity parameter for pruning. A higher value
                       results in more pruning.
        **model_args: Arbitrary keyword arguments to be passed to the
                      DecisionTree constructor (e.g., max_depth, min_samples_split).
    """
    # TODO: Separate features (X) and target (y) from the dataframe.
    
    # TODO: Split the data into training (60%), validation (20%), and test (20%) sets.
    
    # TODO: Instantiate and train the DecisionTree model on the training data.
    
    # TODO: Evaluate the model's accuracy on the test set before pruning.
    
    # TODO: Prune the trained tree using the cost complexity pruning function and the provided alpha.
    
    # TODO: Evaluate the model's accuracy on the test set after pruning.
    
    # TODO: Print the comparison of accuracies before and after pruning.
    

<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h4 style="color: #555;">ارزیابی روی مجموعه داده تایتانیک</h4>
</div>

In [142]:
# TODO


<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h4 style="color: #555;">ارزیابی روی مجموعه داده سرطان پستان</h4>
</div>

In [143]:
# TODO


<div dir="rtl" style="text-align: right; font-family: Arial, sans-serif; line-height: 1.6;">
    <h3 style="color: #2c3e50;">بحث و نتیجه‌گیری نهایی</h3>
    <p>
        با اجرای سلول‌های بالا، می‌توانید تأثیر مستقیم هرس را بر دقت مدل مشاهده کنید. در بسیاری از موارد، به‌ویژه زمانی که درخت اولیه دچار بیش‌برازش شده باشد، دقت مدل بر روی داده‌های آزمون پس از هرس افزایش می‌یابد یا با وجود ساده‌تر شدن مدل، کاهش چشمگیری پیدا نمی‌کند. این نشان می‌دهد که هرس با حذف شاخه‌هایی که فقط نویز داده‌های آموزشی را یاد گرفته‌اند، به تعمیم‌پذیری بهتر مدل کمک می‌کند.
    </p>
    <p>
        انتخاب مقدار بهینه برای <code>alpha</code> معمولاً از طریق اعتبارسنجی متقابل (Cross-validation) انجام می‌شود تا بهترین تعادل بین سادگی و دقت مدل پیدا شود.
    </p>
</div>

<div dir="rtl" style="text-align: right;">

## بخش 5: جنگل تصادفی (Random Forest)
اکنون که درخت تصمیم را داریم، می‌توانیم مدل جنگل تصادفی را بسازیم. این مدل مجموعه‌ای از درختان تصمیم است که هر کدام بر روی یک زیرمجموعه تصادفی از داده‌ها و ویژگی‌ها آموزش دیده‌اند.

### بحث کنید
- **Bagging و Feature Randomness**: جنگل تصادفی چگونه از این دو تکنیک برای کاهش واریانس و بهبود دقت استفاده می‌کند؟

</div>

In [149]:
class RandomForest:
    """
    A Random Forest classifier.
    This version incorporates feature subsampling and uses shallower trees to build
    an ensemble of diverse models.

    Attributes:
        n_trees (int): The number of trees to build in the forest.
        min_samples_split (int): The minimum number of samples required to split an internal node.
        max_depth (int): The maximum depth of each individual tree.
        max_features (str or int): The number of features to consider for each tree.
        trees (list): A list to store the trained DecisionTree objects.
    """
    def __init__(self, n_trees: int = 100, min_samples_split: int = 10, max_depth: int = 10, max_features: Union[str, int] = 'sqrt'):
        """
        Initializes the RandomForest classifier.

        Args:
            n_trees (int): The number of trees to build.
            min_samples_split (int): The minimum number of samples required to split an internal node.
            max_depth (int): The maximum depth of each tree. Defaults to 10 for simpler trees.
            max_features (str or int): The number of features to consider when looking for the best split.
                                     'sqrt' means round(sqrt(n_features)).
                                     'all' means n_features.
        """
        self.n_trees = n_trees
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []

    def _bootstrap_sample(self, X: pd.DataFrame, y: np.ndarray) -> Tuple[pd.DataFrame, np.ndarray]:
        """
        Creates a bootstrap sample of the data (sampling with replacement).

        Args:
            X (pd.DataFrame): The input features.
            y (np.ndarray): The target labels.

        Returns:
            X_sample (pd.DataFrame): A bootstrapped sample of X.
            y_sample (np.ndarray): A bootstrapped sample of y.
        """
        n_samples = X.shape[0]

        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_sample = X.iloc[indices].reset_index(drop=True)
        y_sample = y[indices]
        return X_sample, y_sample

    def fit(self, X: pd.DataFrame, y: np.ndarray) -> None:
        """
        Builds a forest of decision trees from the training set (X, y).

        Args:
            X (pd.DataFrame): The training input samples.
            y (np.ndarray): The target values.
        """
        self.trees = []
        n_features = X.shape[1]
        

        if self.max_features == 'sqrt':
            max_features = int(np.sqrt(n_features))
        elif self.max_features == 'all':
            max_features = n_features
        else:
            max_features = self.max_features


        for _ in range(self.n_trees):
            X_sample, y_sample = self._bootstrap_sample(X, y)

            selected_features = np.random.choice(X.columns, size=max_features, replace=False)
            X_subset = X_sample[selected_features]
            
            tree = DecisionTree(min_samples_split=self.min_samples_split, max_depth=self.max_depth)
            tree.fit(X_subset, y_sample)

            self.trees.append((tree, selected_features))

    def _most_common_label(self, y: np.ndarray) -> Any:
        """
        Finds the most common label in an array of labels.

        Args:
            y (np.ndarray): An array of labels.

        Returns:
            Any: The most frequent label in the array.
        """
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """
        Predicts class labels for samples in X by aggregating predictions from all trees.

        Args:
            X (pd.DataFrame): The input samples to predict.

        Returns:
            np.ndarray: An array of predicted class labels.
        """
        predictions = np.zeros((X.shape[0], self.n_trees), dtype=object)
        for i, (tree, features) in enumerate(self.trees):
            X_subset = X[features]
            predictions[:, i] = tree.predict(X_subset)
        
        final_predictions = np.array([self._most_common_label(row) for row in predictions])
        return final_predictions

<div dir="rtl" style="text-align: right;">

### ارزیابی جنگل تصادفی
این بار مدل `RandomForest` را ارزیابی کنید.

</div>

In [152]:
# TODO: call the function on titanic dataset with RandomForest model
train_and_evaluate(df, 'Survived', RandomForest, n_trees=200, min_samples_split=10, max_depth=8, max_features='all')

--- Results for RandomForest ---
Model Arguments:
  n_trees: 200
  min_samples_split: 10
  max_depth: 8
  max_features: all
Accuracy: 0.8161



In [153]:
# TODO: call the function on breast cancer dataset
train_and_evaluate(df_B, 'is benign', RandomForest, n_trees=150, min_samples_split=10, max_depth=7, max_features='sqrt')

--- Results for RandomForest ---
Model Arguments:
  n_trees: 150
  min_samples_split: 10
  max_depth: 7
  max_features: sqrt
Accuracy: 0.9720



<div dir="rtl" style="text-align: right;">

## بخش 5.5: بحث و نتیجه‌گیری

### بحث کنید
- نتایج دقت دو مدل را روی هر مجموعه داده مقایسه کنید. کدام مدل عملکرد بهتری داشت و چرا؟
- تأثیر هایپرپارامترهایی مانند `max_depth`، `n_trees` و `max_features` بر عملکرد مدل‌ها چیست؟
- چه مزایا و معایب دیگری برای هر یک از این دو الگوریتم می‌توانید نام ببرید؟

</div>